In [1]:
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

I0000 00:00:1785495030.872375   35223 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785495030.934427   35223 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785495032.372343   35223 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
train_data = pd.read_csv("../dataset/train_metadata.csv")
val_data = pd.read_csv("../dataset/val_metadata.csv")
test_data = pd.read_csv("../dataset/test_metadata.csv")

In [3]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [4]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [5]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [6]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

E0000 00:00:1785495044.824207   35223 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomContrast(0.1)

])

In [8]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [9]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [10]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [11]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [12]:
from tensorflow.keras import layers, models

cnn_dropout = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

cnn_dropout.summary()

W0000 00:00:1785407561.959990  842139 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1785407562.012380  842139 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1785407562.026460  842139 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,785,415 (98.36 MB)

 Trainable params: 25,785,415 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
cnn_dropout.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [14]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [15]:
history_basic = cnn_dropout.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
W0000 00:00:1785407566.534141  842139 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1785407571.215953  842280 cpu_allocator_impl.cc:82] Allocation of 205520896 exceeds 10% of free system memory.


220/220 ━━━━━━━━━━━━━━━━━━━━ 396s 2s/step - accuracy: 0.3281 - loss: 1.9313 - val_accuracy: 0.4621 - val_loss: 1.7597
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 373s 2s/step - accuracy: 0.4310 - loss: 1.8055 - val_accuracy: 0.5692 - val_loss: 1.4889
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 386s 2s/step - accuracy: 0.4277 - loss: 1.7494 - val_accuracy: 0.5353 - val_loss: 1.4846
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 383s 2s/step - accuracy: 0.4475 - loss: 1.6139 - val_accuracy: 0.5113 - val_loss: 1.4209
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 446s 2s/step - accuracy: 0.4633 - loss: 1.5069 - val_accuracy: 0.5253 - val_loss: 1.3078
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 446s 2s/step - accuracy: 0.4772 - loss: 1.4071 - val_accuracy: 0.4907 - val_loss: 1.3694
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 445s 2s/step - accuracy: 0.4806 - loss: 1.3729 - val_accuracy: 0.5067 - val_loss: 1.3382
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 450s 2s/step - accuracy: 0.5001 - loss: 1.3405 - val_accuracy: 0.480

In [16]:
train_loss, train_accuracy = cnn_dropout.evaluate(train_dataset)
val_loss, val_accuracy = cnn_dropout.evaluate(val_dataset)
test_loss, test_accuracy = cnn_dropout.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 120s 518ms/step - accuracy: 0.5539 - loss: 1.1060
47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 452ms/step - accuracy: 0.5419 - loss: 1.1353
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 530ms/step - accuracy: 0.5256 - loss: 1.1443


In [17]:
cnn_dropout.save("../models/cnn_dropout.keras")

In [18]:
from tensorflow.keras import layers, models

cnn_dropout_sgd = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

cnn_dropout_sgd.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,785,415 (98.36 MB)

 Trainable params: 25,785,415 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
cnn_dropout_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_cnn_dropout_sgd = cnn_dropout_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 422s 2s/step - accuracy: 0.0839 - loss: 1.9658 - val_accuracy: 0.0113 - val_loss: 1.9578
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 366s 2s/step - accuracy: 0.1347 - loss: 1.9548 - val_accuracy: 0.0120 - val_loss: 1.9456
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 355s 2s/step - accuracy: 0.1399 - loss: 1.9483 - val_accuracy: 0.0260 - val_loss: 1.9384
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 349s 2s/step - accuracy: 0.1399 - loss: 1.9449 - val_accuracy: 0.0226 - val_loss: 1.9469
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 343s 2s/step - accuracy: 0.1566 - loss: 1.9415 - val_accuracy: 0.0679 - val_loss: 1.9404
Restoring model weights from the end of the best epoch: 3.


In [21]:
train_loss, train_accuracy = cnn_dropout_sgd.evaluate(train_dataset)
val_loss, val_accuracy = cnn_dropout_sgd.evaluate(val_dataset)
test_loss, test_accuracy = cnn_dropout_sgd.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 112s 482ms/step - accuracy: 0.0318 - loss: 1.9379
47/47 ━━━━━━━━━━━━━━━━━━━━ 19s 394ms/step - accuracy: 0.0260 - loss: 1.9384
47/47 ━━━━━━━━━━━━━━━━━━━━ 19s 399ms/step - accuracy: 0.0200 - loss: 1.9395


In [22]:
cnn_dropout.save("../models/cnn_dropout_sgd.keras")

In [13]:
from tensorflow.keras import layers, models

cnn_dropout_RMSprop = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

cnn_dropout_RMSprop.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,785,415 (98.36 MB)

 Trainable params: 25,785,415 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
cnn_dropout_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_cnn_dropout_RMSprop = cnn_dropout_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 627s 3s/step - accuracy: 0.3897 - loss: 2.2018 - val_accuracy: 0.3908 - val_loss: 1.7952
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 772s 3s/step - accuracy: 0.4752 - loss: 1.8230 - val_accuracy: 0.4387 - val_loss: 1.5332
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 759s 3s/step - accuracy: 0.4563 - loss: 1.7869 - val_accuracy: 0.4754 - val_loss: 1.4362
Epoch 4/5


I0000 00:00:1785421335.847542  104459 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 971 of 1000
I0000 00:00:1785421336.158537  104459 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 743s 3s/step - accuracy: 0.4622 - loss: 1.7612 - val_accuracy: 0.4680 - val_loss: 1.4577
Epoch 5/5


I0000 00:00:1785422137.896465  121084 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 845 of 1000
I0000 00:00:1785422139.875822  121084 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 755s 3s/step - accuracy: 0.4621 - loss: 1.7149 - val_accuracy: 0.4048 - val_loss: 1.6227
Restoring model weights from the end of the best epoch: 3.


In [16]:
train_loss, train_accuracy = cnn_dropout_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = cnn_dropout_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = cnn_dropout_RMSprop.evaluate(test_dataset)

I0000 00:00:1785422835.685649  137191 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 843 of 1000
I0000 00:00:1785422837.015914  137191 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 860ms/step - accuracy: 0.5076 - loss: 1.4000
47/47 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.4754 - loss: 1.4362
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 842ms/step - accuracy: 0.4844 - loss: 1.4513


In [17]:
cnn_dropout_RMSprop.save("../models/cnn_dropout_RMSprop.keras")

In [21]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [22]:
from tensorflow.keras import layers, models

cnn_dropout_64 = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

cnn_dropout_64.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,785,415 (98.36 MB)

 Trainable params: 25,785,415 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
cnn_dropout_64.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_cnn_dropout_64= cnn_dropout_64.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 703s 6s/step - accuracy: 0.1418 - loss: 2.2720 - val_accuracy: 0.4281 - val_loss: 1.9318
Epoch 2/5


I0000 00:00:1785424289.940379  175340 shuffle_dataset_op.cc:453] ShuffleDatasetV3:32: Filling up shuffle buffer (this may take a while): 880 of 1000
I0000 00:00:1785424291.237198  175340 shuffle_dataset_op.cc:483] Shuffle buffer filled.


110/110 ━━━━━━━━━━━━━━━━━━━━ 681s 6s/step - accuracy: 0.4916 - loss: 1.9548 - val_accuracy: 0.6245 - val_loss: 1.8859
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 697s 6s/step - accuracy: 0.5347 - loss: 1.8935 - val_accuracy: 0.5027 - val_loss: 1.6298
Epoch 4/5


I0000 00:00:1785425667.123429  216625 shuffle_dataset_op.cc:453] ShuffleDatasetV3:32: Filling up shuffle buffer (this may take a while): 893 of 1000
I0000 00:00:1785425668.392855  216625 shuffle_dataset_op.cc:483] Shuffle buffer filled.


110/110 ━━━━━━━━━━━━━━━━━━━━ 703s 6s/step - accuracy: 0.4521 - loss: 1.8196 - val_accuracy: 0.4587 - val_loss: 1.5636
Epoch 5/5


I0000 00:00:1785426370.294635  232741 shuffle_dataset_op.cc:453] ShuffleDatasetV3:32: Filling up shuffle buffer (this may take a while): 806 of 1000
I0000 00:00:1785426372.228067  232741 shuffle_dataset_op.cc:483] Shuffle buffer filled.


110/110 ━━━━━━━━━━━━━━━━━━━━ 683s 6s/step - accuracy: 0.4437 - loss: 1.7867 - val_accuracy: 0.5166 - val_loss: 1.3966
Restoring model weights from the end of the best epoch: 5.


In [24]:
train_loss, train_accuracy = cnn_dropout_64.evaluate(train_dataset)
val_loss, val_accuracy = cnn_dropout_64.evaluate(val_dataset)
test_loss, test_accuracy = cnn_dropout_64.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.5372 - loss: 1.3813
24/24 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.5166 - loss: 1.3966
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.5176 - loss: 1.4203


In [26]:
cnn_dropout_64.save("../models/cnn_dropout_64.keras")

In [12]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

num_classes = 7

def build_model(hp):

    final_model = models.Sequential([
        layers.Input(shape=(224,224,3)),
        layers.Conv2D(
            filters=hp.Choice("filters_1",[32,64]),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(
            filters=hp.Choice("filters_2",[64,128]),
            kernel_size=(3,3),
            activation="relu"
        ),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(
            units=hp.Choice("dense_units",[64,128]),
            activation="relu"
        ),
        layers.Dense(num_classes,activation="softmax")

    ])


    learning_rate = hp.Choice(

        "learning_rate",

        [1e-2,1e-3,1e-4]
    )


    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]
    )


    if optimizer == "adam":

        opt = Adam(
            learning_rate=learning_rate
        )

    
    else:

        opt = RMSprop(
            learning_rate=learning_rate
        )


    final_model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]
    )

    return final_model

In [13]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=3,

    overwrite=True,

    directory="tuner",

    project_name="basic_cnn"
)

In [14]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Trial 3 Complete [00h 31m 12s]
val_accuracy: 0.6697736382484436

Best val_accuracy So Far: 0.6697736382484436
Total elapsed time: 01h 25m 24s


In [15]:
best_dropout = tuner.get_best_models(1)[0]

/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [16]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'filters_1': 64, 'filters_2': 64, 'dense_units': 64, 'learning_rate': 0.01, 'optimizer': 'rmsprop'}


In [13]:
from tensorflow.keras import layers, models

cnn_dropout_final = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

cnn_dropout_final.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,211,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,287,431 (12.54 MB)

 Trainable params: 3,287,431 (12.54 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
cnn_dropout_final.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.01
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_cnn_dropout_64= cnn_dropout_final.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 441s 2s/step - accuracy: 0.0749 - loss: 21.4459 - val_accuracy: 0.1099 - val_loss: 1.8782
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 408s 2s/step - accuracy: 0.2094 - loss: 2.5061 - val_accuracy: 0.0513 - val_loss: 1.9552
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 434s 2s/step - accuracy: 0.0867 - loss: 1.9561 - val_accuracy: 0.0113 - val_loss: 1.9405
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 451s 2s/step - accuracy: 0.0822 - loss: 1.9539 - val_accuracy: 0.6698 - val_loss: 1.8895
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [15]:
train_loss, train_accuracy = cnn_dropout_final.evaluate(train_dataset)
val_loss, val_accuracy = cnn_dropout_final.evaluate(val_dataset)
test_loss, test_accuracy = cnn_dropout_final.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 137s 598ms/step - accuracy: 0.1097 - loss: 1.8782
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 533ms/step - accuracy: 0.1099 - loss: 1.8782
47/47 ━━━━━━━━━━━━━━━━━━━━ 25s 536ms/step - accuracy: 0.1098 - loss: 1.8783


In [16]:
cnn_dropout_final.save("../models/cnn_dropout_final.keras")